In [ ]:
# Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

from src.performance_analysis.density_ratio_analyzer import DensityMapGenerator
from f0_estimator import run_tile_inference
from hedgementation_utils.training.metadata_library import MetadataLibrary, SizeGroup

In [ ]:
# Experiment configuration
DATASET_ROOT = "/scratch/nathan/data/hedgementation_1.3/hedgementation_1.3"
MODEL_DIR = "results/experiment_0"
SAVE_DIR = "results/tile_inference"
RUN_INFERENCE = True
SPLIT_NAME = "infer"
SUBDIVISIONS = 1

# Metadata frame — swap for any GeoDataFrame you like
lib = MetadataLibrary()
all_frames = lib.get_split_metadata(size_group=SizeGroup.SMALL)
metadata_df = all_frames["test"]

distinct_tiles = sorted(all_frames["test"]["tile"].unique().tolist())
tile_remap = {tid: i for i, tid in enumerate(distinct_tiles)}
tile_remap_inv = {i: tid for tid, i in tile_remap.items()}

In [ ]:
# Run inference or reload saved results
if RUN_INFERENCE:
    print(f"Launching tile inference on {len(metadata_df)} patches...")
    generator = run_tile_inference(
        model_dir=MODEL_DIR,
        dataset_root=DATASET_ROOT,
        metadata_df=metadata_df,
        split_name=SPLIT_NAME,
    )
    generator.save(SAVE_DIR)
else:
    print(f"Loading existing results from {SAVE_DIR}...")
    generator = DensityMapGenerator.load(SAVE_DIR, DATASET_ROOT)

In [ ]:
# Accuracy summary
majority_arr = generator.majority_vote[SPLIT_NAME]
labels_arr = generator.labels
entropy_arr = generator.entropy[SPLIT_NAME]

correct = (majority_arr == labels_arr).sum()
total = len(majority_arr)
print(f"Patch-level tile accuracy : {correct}/{total} ({100 * correct / total:.1f}%)")

print("\nPer-tile accuracy:")
for t in sorted(np.unique(labels_arr)):
    mask = labels_arr == t
    c = (majority_arr[mask] == t).sum()
    n = mask.sum()
    orig = t // (SUBDIVISIONS ** 2)
    sub = t % (SUBDIVISIONS ** 2)
    print(f"  bucket {t:3d}  (tile {orig:3d} / sub-tile {sub}) -> {c:3d}/{n:3d}  ({100*c/n:.0f}%)")

In [ ]:
# Map: all patches coloured green (correct) or red (wrong) + tile boundaries
os.makedirs(SAVE_DIR, exist_ok=True)

generator.display_tile_predictions_on_map(
    split_name=SPLIT_NAME,
    save_path=os.path.join(SAVE_DIR, "tile_map.png"),
    tile_remap_inv=tile_remap_inv,
    subdivisions=SUBDIVISIONS,
)

In [ ]:
# Build shared colormap — one fixed colour per tile class across all patches
all_tile_classes = sorted(np.unique(generator.results[SPLIT_NAME]))
cmap_tab = plt.get_cmap("tab20", len(all_tile_classes))
class_to_color = {cls: cmap_tab(i) for i, cls in enumerate(all_tile_classes)}

print(f"Buckets present in predictions: {all_tile_classes}")

In [ ]:
# Most uncertain patch — RGB + pixel tile mask + location on France map
most_uncertain_idx = int(np.argmax(entropy_arr))
patch_id_mu = generator.loaders[SPLIT_NAME].dataset.metadata_frame.iloc[most_uncertain_idx]["ID_PATCH"]

print("-" * 60)
print(f"Most uncertain patch (Index: {most_uncertain_idx} | ID: {patch_id_mu})")
print(f"  Entropy: {entropy_arr[most_uncertain_idx]:.3f}")
print(f"  True bucket: {labels_arr[most_uncertain_idx]} "
      f"(tile {labels_arr[most_uncertain_idx] // (SUBDIVISIONS ** 2)} / "
      f"sub-tile {labels_arr[most_uncertain_idx] % (SUBDIVISIONS ** 2)})")
print(f"  Pred (majority): {majority_arr[most_uncertain_idx]} "
      f"(tile {majority_arr[most_uncertain_idx] // (SUBDIVISIONS ** 2)} / "
      f"sub-tile {majority_arr[most_uncertain_idx] % (SUBDIVISIONS ** 2)})")
print("-" * 60)

generator.display_tile_mask(
    split_name=SPLIT_NAME,
    patch_idx=most_uncertain_idx,
    class_to_color=class_to_color,
    tile_remap_inv=tile_remap_inv,
    subdivisions=SUBDIVISIONS,
    save_path=os.path.join(SAVE_DIR, f"most_uncertain_mask_id_{patch_id_mu}.png"),
)

generator.display_patch_on_france_map(
    split_name=SPLIT_NAME,
    patch_idx=most_uncertain_idx,
    tile_remap_inv=tile_remap_inv,
    subdivisions=SUBDIVISIONS,
    title=f"Most uncertain patch — ID {patch_id_mu}",
    save_path=os.path.join(SAVE_DIR, f"most_uncertain_map_id_{patch_id_mu}.png"),
)

In [ ]:
# Least uncertain patch — RGB + pixel tile mask + location on France map
least_uncertain_idx = int(np.argmin(entropy_arr))
patch_id_lu = generator.loaders[SPLIT_NAME].dataset.metadata_frame.iloc[least_uncertain_idx]["ID_PATCH"]

print("-" * 60)
print(f"Least uncertain patch (Index: {least_uncertain_idx} | ID: {patch_id_lu})")
print(f"  Entropy: {entropy_arr[least_uncertain_idx]:.3f}")
print(f"  True bucket: {labels_arr[least_uncertain_idx]} "
      f"(tile {labels_arr[most_uncertain_idx] // (SUBDIVISIONS ** 2)} / "
      f"sub-tile {labels_arr[most_uncertain_idx] % (SUBDIVISIONS ** 2)})")
print(f"  Pred (majority): {majority_arr[least_uncertain_idx]} "
      f"(tile {majority_arr[most_uncertain_idx] // (SUBDIVISIONS ** 2)} / "
      f"sub-tile {majority_arr[most_uncertain_idx] % (SUBDIVISIONS ** 2)})")
print("-" * 60)

generator.display_tile_mask(
    split_name=SPLIT_NAME,
    patch_idx=least_uncertain_idx,
    class_to_color=class_to_color,
    tile_remap_inv=tile_remap_inv,
    subdivisions=SUBDIVISIONS,
    save_path=os.path.join(SAVE_DIR, f"least_uncertain_mask_id_{patch_id_lu}.png"),
)

generator.display_patch_on_france_map(
    split_name=SPLIT_NAME,
    patch_idx=least_uncertain_idx,
    tile_remap_inv=tile_remap_inv,
    subdivisions=SUBDIVISIONS,
    title=f"Least uncertain patch — ID {patch_id_lu}",
    save_path=os.path.join(SAVE_DIR, f"least_uncertain_map_id_{patch_id_lu}.png"),
)